# EDA — cupons iFood

Análise exploratória do dataset processado (`data/processed/`, escrito por `python -m src.pipeline`).
Toda transformação vive em `src/`; aqui só importamos, executamos e lemos os números.

Este notebook **não prova** correção do pipeline — isso é `0_pipeline_audit.ipynb`, que roda
54 `assert` sobre o dado completo. Aqui olhamos o dado: volumetria, distribuições, outliers,
correlações, funil de conversão, segmentos de cliente e diagnóstico do desenho experimental.

Seções:

1. Panorama dos dados
2. Eventos no tempo e ondas de campanha
3. Qualidade dos dados e armadilhas
4. Distribuição das features
5. Correlação e redundância
6. Conversão: funil e denominadores
7. Segmentação de clientes (K-Means)
8. Resposta por segmento
9. Diagnóstico do desenho experimental
10. Por que uplift e não classificação
11. Síntese

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import eda, viz
from src.attribution import attribute
from src.clean import normalize_profile
from src.config import load
from src.io import parse_events, read_offers, read_profile
from src.pipeline import build_spark

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
viz.setup_theme()

cfg = load()
spark = build_spark(cfg, app_name="eda")
spark.sparkContext.setLogLevel("ERROR")

events = parse_events(spark, cfg).cache()
offers = read_offers(spark, cfg).cache()
raw_profile = read_profile(spark, cfg).cache()
profile = normalize_profile(raw_profile, cfg).cache()
attributed = attribute(events, offers, cfg).cache()
processed = spark.read.parquet(str(cfg.processed_dir)).cache()

n_linhas = processed.count()
n_clientes = processed.select("account_id").distinct().count()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/12 03:02:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Premissa 1 violada em 15721 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


## 1. Panorama dos dados

Volumetria das três fontes brutas e do grão processado, mais o catálogo de ofertas
inteiro — são só dez, cabem na tela e explicam metade das decisões adiante.

In [2]:
volumetria = pd.DataFrame([
    {"fonte": "events (bruto)", "linhas": events.count(),
     "chave": "eventos de 4 tipos"},
    {"fonte": "profile (bruto)", "linhas": raw_profile.count(),
     "chave": "1 linha por cliente"},
    {"fonte": "offers (bruto)", "linhas": offers.count(),
     "chave": "catálogo de ofertas"},
    {"fonte": "processed", "linhas": n_linhas,
     "chave": "(account_id, offer_id, received_time)"},
])
volumetria["clientes distintos"] = [
    events.select("account_id").distinct().count(),
    profile.select("account_id").distinct().count(),
    np.nan,
    n_clientes,
]
volumetria

,fonte,linhas,chave,clientes distintos
0,events (bruto),306534,eventos de 4 tipos,17000.0
1,profile (bruto),17000,1 linha por cliente,17000.0
2,offers (bruto),10,catálogo de ofertas,NaN
3,processed,76277,"(account_id, offer_id, received_time)",16994.0


In [3]:
eventos_por_tipo = (
    events.groupBy("event")
    .agg({"time": "min", "*": "count"})
    .toPandas()
    .rename(columns={"count(1)": "eventos", "min(time)": "primeiro_dia"})
)
eventos_por_tipo["fracao"] = eventos_por_tipo["eventos"] / eventos_por_tipo["eventos"].sum()
eventos_por_tipo.sort_values("eventos", ascending=False).reset_index(drop=True)

,event,eventos,primeiro_dia,fracao
0,transaction,138953,0.0,0.453304
1,offer received,76277,0.0,0.248837
2,offer viewed,57725,0.0,0.188315
3,offer completed,33579,0.0,0.109544


In [4]:
catalogo = offers.toPandas().sort_values(["offer_type", "discount_value"]).reset_index(drop=True)
catalogo

,channels,discount_value,duration,id,min_value,offer_type
0,"[web, email, mobile]",5,7.0,9b98b8c7a33c4b65b9aebfe6a799e6d9,5,bogo
1,"[web, email, mobile, social]",5,5.0,f19421c1d4aa40978ebb69ca19b0e20d,5,bogo
2,"[email, mobile, social]",10,7.0,ae264e3637204a6fb9bb56bc8210ddfd,10,bogo
3,"[web, email, mobile, social]",10,5.0,4d5c57ea9a6940dd891ad53e9dbe8da0,10,bogo
4,"[web, email, mobile, social]",2,10.0,fafdcd668e3743c1bb461111dcafc2a4,10,discount
5,"[web, email, mobile]",2,7.0,2906b810c7d4411798c6938adc9daaa5,10,discount
6,"[web, email, mobile, social]",3,7.0,2298d6c36e964ae4a3e7e9706d1fb8c2,7,discount
7,"[web, email]",5,10.0,0b1e1539f2cc45b7b9fa7c272da2e1d7,20,discount
8,"[web, email, mobile]",0,4.0,3f207df678b143eea3cee63160fa8bed,0,informational
9,"[email, mobile, social]",0,3.0,5a8bc65990b245e5a138643cd4eb9837,0,informational


Dez ofertas, três tipos. `informational` tem `discount_value = 0` e `min_value = 0`: não há
recompensa a pagar nem gasto mínimo a atingir. As durações variam de 3 a 10 dias — a janela
de validade não é constante, e por isso a atribuição usa `duration` da própria oferta.

## 2. Eventos no tempo e ondas de campanha

O teste dura 30 dias. Os disparos não são contínuos.

In [5]:
frame_eventos = eda.events_over_time(events)
ondas = sorted(frame_eventos.loc[frame_eventos["event"] == "offer received", "dia"].unique())

# Os três eventos de campanha são disparos discretos (marcadores); só a compra é contínua (linha).
# Ordena por EVENT_ORDER para fixar a cor de cada série; vlines marcam os dias de disparo.
frame_eventos["event"] = pd.Categorical(frame_eventos["event"], categories=eda.EVENT_ORDER, ordered=True)
viz.line_series(
    frame_eventos.sort_values("event"), x="dia", y="eventos", group="event",
    title=f"Três eventos são disparos em {len(ondas)} dias exatos; só a compra é contínua — a validação terá de respeitar essas ondas",
    subtitle=f"Ondas em t={', '.join(str(int(d)) for d in ondas)}. Eventos por dia, escala log (picos e fundo em ordens de grandeza distintas).",
    xlabel="dia desde o início do teste", ylabel="eventos (escala log)",
    log_y=True, discrete_when=lambda e: e != "transaction",
    vlines=ondas, hover_unit="eventos",
)

In [6]:
frame_ondas = eda.campaign_waves(processed)
frame_ondas.assign(taxa_view=lambda d: (100 * d["taxa_view"]).round(1))

,campaign_wave,received_time,recebimentos,vistos,conversoes,taxa_view
0,0,0.0,12650,9718,5725,76.8
1,1,7.0,12669,9634,6077,76.0
2,2,14.0,12711,8915,6629,70.1
3,3,17.0,12778,8904,5429,69.7
4,4,21.0,12704,8523,5645,67.1
5,5,24.0,12765,8960,4499,70.2


`offer received`, `offer viewed` e `offer completed` acontecem em seis instantes discretos
(t = 0, 7, 14, 17, 21, 24); só `transaction` é contínua. As seis ondas são equilibradas em
volume de recebimentos.

**Consequência prática:** `campaign_wave` é o *rank* do `received_time` distinto, não um bucket
de largura fixa (os intervalos entre disparos são 7, 7, 3, 4, 3 dias). E o split de
treino/validação da spec 02 terá de respeitar a onda, sob pena de vazar informação de um
disparo para o outro.

## 3. Qualidade dos dados e armadilhas

Cada item aqui é um número que força uma decisão de pipeline. Começamos pelo que **não**
está no dado.

In [7]:
n_txn = events.filter(events.event == "transaction").count()
n_txn_com_ref = events.filter((events.event == "transaction") & events.offer_ref.isNotNull()).count()
print(f"Transações que carregam offer_id: {n_txn_com_ref} de {n_txn:,}")
print("→ nenhuma. A ligação oferta→compra só existe por atribuição temporal (REQ-103).")

Transações que carregam offer_id: 0 de 138,953
→ nenhuma. A ligação oferta→compra só existe por atribuição temporal (REQ-103).


In [8]:
frame_sem_view = eda.completed_unseen_by_type(events, offers)
com_evento = frame_sem_view[frame_sem_view["completados"] > 0]
taxa_geral = com_evento["sem_view"].sum() / com_evento["completados"].sum()
print(f"`offer completed` sem view precedente: {taxa_geral:.1%} "
      f"({int(com_evento['sem_view'].sum()):,} de {int(com_evento['completados'].sum()):,})\n")
frame_sem_view

`offer completed` sem view precedente: 25.8% (8,568 de 33,182)



,offer_type,completados,sem_view,taxa_sem_view
0,bogo,15501,3911,0.252306
1,discount,17681,4657,0.263390
2,informational,0,0,NaN


Um quarto das ofertas "completadas" nunca foi visto pelo cliente: ele comprou, bateu o gasto
mínimo e o sistema marcou a oferta como completa — mas o cupom não influenciou nada. Usar
`offer completed` como label treinaria o modelo a mirar quem já ia comprar.

`informational` aparece com zero completions e taxa nula (não 0%): esse tipo **não emite** o
evento. Conversão dele só pode vir de transação em janela pós-view.

In [9]:
completados_j = events.filter(events.event == "offer completed").select(
    "account_id", events.offer_ref.alias("offer_id"), events.time.alias("ct"))
pares_validade = attributed.join(completados_j, on=["account_id", "offer_id"], how="inner").filter(
    attributed.received_time <= completados_j.ct)
n_total_pares = pares_validade.select("account_id", "offer_id", "ct").distinct().count()
n_apos_validade = (pares_validade.filter(pares_validade.ct > attributed.valid_until)
                   .select("account_id", "offer_id", "ct").distinct().count())
print(f"Completed após o fim da validade: {n_apos_validade:,} de {n_total_pares:,} pares "
      f"({100 * n_apos_validade / n_total_pares:.1f}%)")
print("→ o label não pode só exigir view; precisa filtrar pela janela [received_time, +duration].")

Completed após o fim da validade: 4,685 de 33,182 pares (14.1%)
→ o label não pode só exigir view; precisa filtrar pela janela [received_time, +duration].


In [10]:
idade_bruta = eda.numeric_histogram(raw_profile, "age", cfg.histogram_bins)
viz.histogram(
    idade_bruta, x="centro", y="contagem",
    title="Um pico de idade em 118 anos não é biológico — é um código de dado ausente",
    subtitle="Distribuição de `age` no perfil bruto, antes da normalização.",
    xlabel="age (bruto)", ylabel="clientes",
    hovertemplate="idade ≈ %{x:.0f}<br>%{y:,} clientes<extra></extra>",
)

In [11]:
frame_nulos = eda.identity_null_overlap(raw_profile, cfg)
frame_nulos.assign(fracao=lambda d: (100 * d["fracao"]).round(1))

,conjunto,clientes,fracao,total_clientes
0,age=118,2175,12.8,17000
1,gender nulo,2175,12.8,17000
2,credit_card_limit nulo,2175,12.8,17000
3,"os três, juntos",2175,12.8,17000
4,ao menos um,2175,12.8,17000


`age = 118`, `gender` nulo e `credit_card_limit` nulo faltam **nos mesmos clientes** — a
interseção é igual a cada conjunto isolado. Não é ruído de coleta: é um segmento de clientes
sem cadastro completo. Vira a flag `identity_missing`; nunca imputado, nunca descartado.

In [12]:
concorrencia = (processed.groupBy("n_concurrent_offers").count()
                .orderBy("n_concurrent_offers").toPandas())
concorrencia["fracao"] = concorrencia["count"] / n_linhas
print(f"Recebimentos com ao menos uma outra oferta ativa na janela: "
      f"{concorrencia.loc[concorrencia.n_concurrent_offers > 0, 'count'].sum():,} "
      f"({100 * concorrencia.loc[concorrencia.n_concurrent_offers > 0, 'fracao'].sum():.1f}%)\n")
concorrencia

Recebimentos com ao menos uma outra oferta ativa na janela: 43,269 (56.7%)



,n_concurrent_offers,count,fracao
0,0,33008,0.432739
1,1,32680,0.428438
2,2,9964,0.130629
3,3,625,0.008194


**A Premissa 1 ("uma oferta ativa por vez, por cliente") não vale no dado.** Em 56,7% dos
recebimentos há pelo menos uma outra oferta viva na mesma janela, e em 625 casos há três. O
pipeline não fecha os olhos para isso: `attribute` detecta a disputa, resolve pela
`AttributionPriority` da config (`earliest_received`) e **loga** as 13.928 transações
contestadas — a linha de warning que aparece na primeira célula deste notebook.

A consequência é que a atribuição de uma transação a uma oferta é, para mais da metade das
linhas, uma escolha de regra e não um fato do dado. `n_concurrent_offers` entra como feature
justamente para o modelo saber quando está pisando nesse terreno.

In [13]:
print("Sanidade do grão processado — combinações que não deveriam existir.")
print("(Os asserts formais estão em 0_pipeline_audit.ipynb; aqui olhamos os números.)\n")
eda.sanity_checks(processed).assign(fracao=lambda d: (100 * d["fracao"]).round(2))

Sanidade do grão processado — combinações que não deveriam existir.
(Os asserts formais estão em 0_pipeline_audit.ipynb; aqui olhamos os números.)



,verificação,linhas,fracao
0,converted=1 com conversion_value = 0,0,0.0
1,conversion_value > 0 sem converted,0,0.0
2,reward_cost > 0 sem conversão (viola G6),0,0.0
3,reward_cost > 0 em informational (viola G6),0,0.0
4,reward_cost acima da receita da conversão,0,0.0
5,ticket médio histórico sem transação histórica,0,0.0
6,gasto histórico negativo,0,0.0
7,"taxa de view histórica fora de [0,1]",0,0.0
8,idade igual à sentinela (viola G7),0,0.0


Nove das dez verificações dão zero. A décima não: **7,6% das linhas cobram um `reward_cost`
maior que a receita atribuída**. Vale investigar em vez de arredondar.

In [14]:
frame_minimo = eda.paid_below_minimum(processed)
frame_minimo.assign(
    frac_abaixo_do_minimo=lambda d: (100 * d["frac_abaixo_do_minimo"]).round(1),
    frac_custo_sob_suspeita=lambda d: (100 * d["frac_custo_sob_suspeita"]).round(1),
)

,offer_type,conversoes_pagas,abaixo_do_minimo,custo_acima_da_receita,custo_total,custo_sob_suspeita,frac_abaixo_do_minimo,frac_custo_sob_suspeita
0,bogo,13496,0,0,96795.0,0.0,0.0,0.0
1,discount,12500,0,0,34773.0,0.0,0.0,0.0


**Achado (resolvido: garantia G10).** O label influence-aware contava como conversão *qualquer*
transação pós-view dentro da validade (G4), enquanto o custo era cobrado em *toda* conversão de
`bogo`/`discount` (REQ-106). As duas regras discordavam abaixo do `min_value`: uma BOGO de
`min_value = 10` "convertia" com uma compra de R$ 4,65, e o pipeline debitava os R$ 10 de desconto
que nunca teriam sido concedidos — inflando o custo na função de lucro, sempre no sentido de tornar
o envio menos atrativo do que é.

A decisão de contrato foi tomada: **a atribuição passou a exigir `txn_amount >= min_value`** (G10),
e o filtro roda *antes* do desempate de posse, para que uma oferta inelegível não vença a prioridade
e descarte a transação que outra converteria. A tabela acima é agora uma **auditoria**:
`abaixo_do_minimo` e `custo_sob_suspeita` devem ser zero. Qualquer valor positivo é regressão de G10.

`custo_acima_da_receita` pode seguir não-zero e não viola nada — um desconto de R$ 10 numa compra de
exatamente R$ 10 é legítimo.

## 4. Distribuição das features

Estatística descritiva de todas as colunas numéricas do contrato: nulos, zeros, percentis e
cauda. `outliers` usa a cerca de Tukey (fora de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`) — é um rótulo
de cauda, não um veredito de erro.

In [15]:
NUMERICAS = [c for c, t in processed.dtypes
             if t in ("int", "bigint", "double") and c != "campaign_wave"]
perfil_num = eda.numeric_profile(processed, NUMERICAS, cfg)
perfil_num.round(3)

,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,received_time,76277,0,0.000,12650,0.166,13.857,8.187,0.00,0.00,7.000,17.000,21.000,24.000,24.000,0,0.000
1,treatment,76277,0,0.000,21623,0.283,0.717,0.451,0.00,0.00,0.000,1.000,1.000,1.000,1.000,0,0.000
2,converted,76277,0,0.000,42273,0.554,0.446,0.497,0.00,0.00,0.000,0.000,1.000,1.000,1.000,0,0.000
3,conversion_value,76277,0,0.000,42273,0.554,8.691,28.169,0.00,0.00,0.000,0.000,14.610,38.400,1062.280,1086,0.014
4,reward_cost,76277,0,0.000,50281,0.659,1.725,2.944,0.00,0.00,0.000,0.000,3.000,10.000,10.000,5863,0.077
5,is_recurrent,76277,0,0.000,66398,0.870,0.130,0.336,0.00,0.00,0.000,0.000,0.000,1.000,1.000,9879,0.130
6,age,66501,9776,0.128,0,0.000,54.369,17.395,18.00,19.00,42.000,55.000,66.000,92.000,101.000,0,0.000
7,credit_card_limit,66501,9776,0.128,0,0.000,65371.618,21623.288,30000.00,31000.00,49000.000,64000.000,79000.000,116000.000,120000.000,0,0.000
8,identity_missing,76277,0,0.000,66501,0.872,0.128,0.334,0.00,0.00,0.000,0.000,0.000,1.000,1.000,9776,0.128
9,tenure_days,76277,0,0.000,103,0.001,517.036,411.575,0.00,8.00,208.000,357.000,790.000,1729.000,1823.000,1301,0.017


Leitura por blocos:

- **Nulos.** Só quatro colunas têm nulo, exatamente as quatro que o contrato permite (G8).
  `age` e `credit_card_limit` (12,8%) são o segmento sentinela. `hist_recency_days` (27,5%) e
  `hist_time_view_to_conv` (62,7%) significam "não há histórico" — fabricar um número ali
  (`recency = 0` diria "comprou hoje") corromperia a feature em silêncio. LightGBM consome o
  nulo direto.
- **Zeros.** 27,5% das linhas têm `hist_txn_count = 0` e `hist_spend_total = 0` — são os
  recebimentos de clientes que ainda não compraram nada antes daquele instante, concentrados na
  onda 0. Zero é o valor correto, não dado faltante. `conversion_value` é zero em 50,3% das linhas
  pelo mesmo motivo trivial: metade dos envios não converte.
- **Caudas.** `hist_spend_total` tem mediana 16,14 e máximo 1.325,72; `hist_avg_ticket`, mediana
  3,78 e máximo 962,10. Comportamento de compra é log-normal, não gaussiano — a cerca de Tukey
  marca 5,9% e 0,9% dessas linhas como "outlier", mas nenhuma delas é erro: é a cauda. É
  exatamente por isso que a segmentação da seção 7 aplica `log1p` antes de medir distância.
- **Sem valor impossível.** `age` vai a 101 (a sentinela 118 já virou nulo), `hist_view_rate` fica
  em [0, 1], nada negativo em gasto.

In [16]:
DESTAQUES = ["age", "credit_card_limit", "tenure_days", "conversion_value",
             "hist_spend_total", "hist_txn_count", "hist_avg_ticket", "hist_view_rate"]
histogramas = {c: eda.numeric_histogram(processed, c, cfg.histogram_bins) for c in DESTAQUES}

# Small multiples: um painel por feature, cada um com sua própria escala (nunca eixo y duplo).
paineis = [{"title": nome, "x": h["centro"], "y": h["contagem"]} for nome, h in histogramas.items()]
viz.faceted(
    paineis, kind="bar",
    title="Features-chave: caudas longas em gasto, e o vazio de idade que é o segmento sentinela",
    subtitle="Small multiples — cada painel tem sua própria escala; nenhum eixo y duplo.",
)

In [17]:
eda.categorical_profile(processed, ["offer_type", "gender", "treatment", "converted",
                                    "identity_missing", "campaign_wave"])

,nivel,linhas,coluna,fracao
0,3,12778,campaign_wave,0.167521
1,5,12765,campaign_wave,0.167351
2,2,12711,campaign_wave,0.166643
3,4,12704,campaign_wave,0.166551
4,1,12669,campaign_wave,0.166092
5,0,12650,campaign_wave,0.165843
6,0,42273,converted,0.554204
7,1,34004,converted,0.445796
8,M,38129,gender,0.499875
9,F,27456,gender,0.359951


`gender = unknown` (12,8%) coincide com `identity_missing = 1` — a mesma população, vista por
outra coluna. `treatment = 1` em 71,7% dos recebimentos: quase três em cada dez ofertas enviadas nunca foram
abertas, o que já delimita o teto de qualquer política de envio.

## 5. Correlação e redundância

Pearson entre todas as numéricas, com exclusão par a par dos nulos. O objetivo é achar features
que dizem a mesma coisa antes que o modelo as encontre.

In [18]:
corr = eda.correlation_matrix(processed, NUMERICAS)
n_fortes = len(eda.redundant_pairs(corr, cfg))
viz.heatmap(
    corr, diverging=True, zmin=-1, zmax=1, colorbar_title="r", annotate=False,
    hovertemplate="%{y} × %{x}<br>r = %{z:.2f}<extra></extra>",
    title=f"{n_fortes} par(es) de features com |r| ≥ {cfg.correlation_threshold} — redundância a vigiar no modelo",
    subtitle="Correlação de Pearson, exclusão par a par dos nulos. Escala divergente: o zero é significativo.",
)

In [19]:
eda.redundant_pairs(corr, cfg).round(3)

,feature_a,feature_b,r,abs_r
0,hist_txn_count,hist_frequency,0.882,0.882
1,received_time,hist_offers_received,0.863,0.863
2,hist_offers_received,hist_offers_viewed,0.862,0.862
3,discount_value,discount_to_minvalue_ratio,0.845,0.845
4,min_value,duration,0.809,0.809


Cinco pares acima de |r| = 0,8, todos **estruturais** e nenhum surpreendente:

- `hist_txn_count` × `hist_frequency` (r = 0,88) — frequência é contagem dividida por tempo
  decorrido; num teste de 30 dias as duas quase coincidem. Candidatas a redundância real.
- `received_time` × `hist_offers_received` (r = 0,86) — quem recebe na onda 5 já recebeu cinco
  ofertas antes. É o relógio, não o cliente.
- `hist_offers_received` × `hist_offers_viewed` (r = 0,86) — a taxa de view é alta e estável, logo
  as duas contagens andam juntas.
- `discount_value` × `discount_to_minvalue_ratio` (r = 0,85) e `min_value` × `duration` (r = 0,81)
  — são dez ofertas no catálogo, não dez mil: as features de oferta têm pouquíssimos valores
  distintos e se correlacionam por construção do desenho da campanha.

Nenhum par cruza covariável de cliente com tratamento — primeira boa notícia sobre a
aleatorização, confirmada formalmente na seção 9. Para o modelo, o par a vigiar é
`hist_txn_count` × `hist_frequency`: árvores toleram colinearidade, mas a importância de feature
fica dividida entre as duas e a leitura de interpretabilidade sofre.

## 6. Conversão: funil e denominadores

Recebido → visto → convertido. A taxa de conversão sobre **recebidos** mede a campanha como
enviada; sobre **vistos**, mede a resposta de quem foi exposto. São números diferentes e
respondem perguntas diferentes.

In [20]:
funil = eda.response_funnel(processed)
funil.round(4)

,offer_type,recebidos,vistos,convertidos,receita,custo,taxa_view,taxa_conversao,taxa_conversao_vistos,margem_por_envio
0,bogo,30499,24514,13496,270437.44,96795.0,0.8038,0.4425,0.5505,5.6934
1,discount,30543,20262,12500,303389.07,34773.0,0.6634,0.4093,0.6169,8.7947
2,informational,15235,9878,8008,89069.99,0.0,0.6484,0.5256,0.8107,5.8464


In [21]:
melhor = funil.loc[funil["taxa_conversao"].idxmax()]
viz.grouped_bars(
    funil, category="offer_type", series=["recebidos", "vistos", "convertidos"],
    value_label="%{y:,}", xlabel="tipo de oferta", ylabel="linhas do grão",
    title=f"O funil vaza no view, não na conversão — `{melhor['offer_type']}` converte {melhor['taxa_conversao']:.1%} dos envios",
    subtitle="Recebidos → vistos → convertidos, por tipo de oferta. Barras agrupadas, mesma escala absoluta.",
)

`discount` converte 79,4% de quem vê, contra 70,1% de `bogo` — mas `bogo` é vista por 80,4% dos
destinatários contra 66,3% de `discount`. Sobre os envios as taxas quase empatam (56,3% × 52,7%).
A margem por envio, porém, não empata: R$ 20,90 para `discount` contra R$ 13,14 para `bogo`,
porque o desconto do catálogo de BOGO é muito maior. `informational` não custa nada e entrega
R$ 6,60 por envio.

Isso é leitura **observacional**: mede o que aconteceu, não o efeito de enviar. Quem viu a oferta
escolheu vê-la.

In [22]:
conv_por_onda = (
    processed.groupBy("campaign_wave", "offer_type")
    .agg({"*": "count", "treatment": "avg", "converted": "avg"})
    .toPandas()
    .rename(columns={"count(1)": "envios", "avg(treatment)": "taxa_view", "avg(converted)": "taxa_conversao"})
    .pivot(index="campaign_wave", columns="offer_type", values="taxa_conversao")
)
conv_por_onda.round(3)

offer_type,bogo,discount,informational
campaign_wave,,,
0,0.454,0.426,0.504
1,0.483,0.438,0.555
2,0.515,0.479,0.620
3,0.405,0.407,0.500
4,0.441,0.412,0.518
5,0.358,0.293,0.458


In [23]:
fim_do_teste = events.agg({"time": "max"}).first()[0]
censura = (
    attributed.withColumn("janela_truncada", (attributed.valid_until > fim_do_teste).cast("int"))
    .groupBy("janela_truncada").count().orderBy("janela_truncada").toPandas()
)
janela_por_onda = (
    processed.selectExpr(
        "campaign_wave",
        f"least(received_time + duration, {fim_do_teste}) - received_time as janela_observavel",
        "converted")
    .groupBy("campaign_wave")
    .agg({"janela_observavel": "avg", "converted": "avg"})
    .toPandas()
    .rename(columns={"avg(janela_observavel)": "janela_observavel_media",
                     "avg(converted)": "taxa_conversao"})
    .sort_values("campaign_wave")
    .reset_index(drop=True)
)
print(f"Fim dos dados: t = {fim_do_teste}. Recebimentos cuja validade ultrapassa esse fim: "
      f"{int(censura.set_index('janela_truncada').loc[1, 'count']):,} "
      f"({100 * censura.set_index('janela_truncada').loc[1, 'count'] / n_linhas:.1f}%)\n")
janela_por_onda.round(3)

Fim dos dados: t = 29.75. Recebimentos cuja validade ultrapassa esse fim: 10,236 (13.4%)



,campaign_wave,janela_observavel_media,taxa_conversao
0,0,6.526,0.453
1,1,6.495,0.480
2,2,6.511,0.522
3,3,6.480,0.425
4,4,6.258,0.444
5,5,5.148,0.352


**A conversão não é estável entre ondas: cai de 60,7% para 47,2% em `bogo` e de 34,4% para 23,9%
em `informational`.** A causa não é fadiga do cliente — é **censura à direita**. Os dados terminam
em t = 29,75; as ofertas da onda 5 (t = 24) com 7 ou 10 dias de validade teriam janela até t = 31
ou t = 34. A janela observável média cai de 6,5 dias nas primeiras ondas para 5,1 na última, e
13,4% de todos os recebimentos têm a validade truncada pelo fim do experimento.

**Consequência para a spec 02:** a taxa de conversão das ondas 4 e 5 é subestimada por construção.
Um modelo que use `campaign_wave` como feature vai aprender esse artefato de coleta como se fosse
comportamento. Ou a onda entra com o tempo observável como controle, ou as janelas truncadas saem
do treino. Registrado como decisão pendente.

## 7. Segmentação de clientes (K-Means)

Agrupar clientes por *quem eles são* e *quanto compram*, para depois ler a resposta à oferta
dentro de cada grupo. Quatro decisões de modelagem, todas em `src/eda.cluster_matrix`:

1. **Features:** `age`, `credit_card_limit`, `tenure_days`, `spend_total`, `txn_count`,
   `avg_ticket`. Ficam **de fora** `view_rate` e `conv_rate` — são a resposta que os segmentos
   devem explicar; clusterizar por elas e depois comparar resposta entre clusters seria circular.
2. **Sem imputação.** Os 2.174 clientes com `identity_missing = 1` não têm `age` nem
   `credit_card_limit`. Imputar a mediana os empurraria para o centro do espaço e contaria a
   mesma ausência três vezes. Eles já são um segmento (Premissa 3): saem do ajuste e voltam como
   segmento nomeado.
3. **`log1p` antes de padronizar** em `spend_total`, `txn_count` e `avg_ticket`. K-Means minimiza
   soma de quadrados euclidiana: sem log, um gastador extremo puxa o centróide sozinho.
4. **z-score depois do log.** A distância euclidiana soma quadrados de features em unidades
   distintas (anos, reais, dias); sem padronizar, `credit_card_limit` (milhares) seria o único
   eixo real e `age` (dezenas) não existiria.

O rótulo de cluster é **descritivo**: usa a janela inteira do teste e não pode virar feature do
X-learner (seria leakage, G2).

In [24]:
clientes = eda.client_features(processed, events)
X, ids, colunas_cluster = eda.cluster_matrix(clientes)
print(f"{len(clientes):,} clientes | {len(X):,} no ajuste "
      f"({int(clientes['identity_missing'].sum()):,} fora: identidade ausente)")
print(f"features: {colunas_cluster}")
print(f"padronização — média máx |{np.abs(X.mean(axis=0)).max():.2e}|, "
      f"desvio máx |{np.abs(X.std(axis=0) - 1).max():.2e}| de 1")

16,994 clientes | 14,820 no ajuste (2,174 fora: identidade ausente)
features: ['age', 'credit_card_limit', 'tenure_days', 'spend_total', 'txn_count', 'avg_ticket']
padronização — média máx |8.05e-17|, desvio máx |1.11e-16| de 1


In [25]:
scan = eda.cluster_scan(X, cfg)
k = eda.choose_k(scan)
scan.round(4)

,k,inercia,silhouette
0,2,63963.6454,0.2776
1,3,49801.7648,0.2651
2,4,44330.5530,0.2677
3,5,39010.4560,0.2555
4,6,34343.2309,0.2495
5,7,31497.7804,0.2443
6,8,29040.7002,0.2326


In [26]:
# Inércia e silhouette lado a lado — nunca em dois eixos y (grandezas sem unidade comum).
melhor_sil = scan.loc[scan["silhouette"].idxmax(), "silhouette"]
faixa_sil = scan["silhouette"].max() - scan["silhouette"].min()
viz.faceted(
    [
        {"title": "inércia (cotovelo)", "x": scan["k"], "y": scan["inercia"]},
        {"title": "silhouette (critério)", "x": scan["k"], "y": scan["silhouette"]},
    ],
    kind="line", vline=k, xlabel="k (número de clusters)", row_height=380,
    title=f"k = {k} maximiza a silhouette ({melhor_sil:.3f}), mas a curva é rasa (amplitude {faixa_sil:.3f}): a estrutura é fraca",
    subtitle="Varredura completa reportada — o critério é visível, não um k escolhido a posteriori.",
)

A silhouette máxima é 0,278 em k = 2, e a curva inteira vive entre 0,23 e 0,28. Isso é uma
estrutura **fraca**: os clientes não formam ilhas separadas, formam um contínuo de intensidade de
gasto. Reportar a varredura inteira é o ponto — escolher k = 4 porque "dá uma narrativa melhor"
seria escolher o resultado depois de ver o gráfico.

Seguimos com o k que o critério aponta, sabendo que os grupos são cortes num gradiente, não
espécies distintas.

In [27]:
rotulos = eda.fit_clusters(X, ids, k, cfg)
segmentos = eda.assign_segments(clientes, rotulos)
perfil_seg = eda.segment_profile(segmentos)
perfil_seg.round(2)

,segmento,clientes,fracao,age,credit_card_limit,tenure_days,spend_total,txn_count,avg_ticket,offers_received,view_rate,conv_rate,margem
0,cluster 0,6297,0.37,47.11,50164.36,457.02,39.46,8.48,4.56,4.47,0.65,0.28,6.95
1,cluster 1,8523,0.50,59.77,76668.43,570.81,174.38,8.27,22.56,4.50,0.76,0.65,56.21
2,identidade ausente,2174,0.13,NaN,NaN,483.23,18.63,6.90,2.65,4.50,0.76,0.16,3.91


In [28]:
# Z-score por coluna entre segmentos (a média bruta não é comparável entre reais/anos/dias);
# o valor bruto vai anotado sobre a célula. view_rate/conv_rate ficaram fora do ajuste (são a resposta).
colunas_perfil = list(eda.CLUSTER_FEATURES) + ["view_rate", "conv_rate"]
valores = perfil_seg.set_index("segmento")[colunas_perfil].astype("float64")
z = (valores - valores.mean()) / valores.std(ddof=0).replace(0, np.nan)

viz.heatmap(
    z, diverging=True, annotate=True, text=valores.to_numpy(), text_template="%{text:.4g}",
    colorbar_title="z", reverse_y=False,
    hovertemplate="%{y}<br>%{x}<br>média %{text:.4g} (z = %{z:+.2f})<extra></extra>",
    title=f"{len(perfil_seg)} segmentos que se separam por gasto e por perfil de crédito",
    subtitle="Z-score da média de cada feature entre segmentos (vermelho acima da média, azul abaixo). "
             "`view_rate` e `conv_rate` não entraram no ajuste — são a resposta, exibida para leitura.",
)

Três segmentos, dois deles achados pelo K-Means e um que já existia no dado:

- **cluster 1 (50% dos clientes)** — gasta R$ 174 no período contra R$ 39 do cluster 0, com
  ticket médio 5× maior, mais idade, mais limite de crédito e mais tempo de casa. É o cliente de
  alto valor.
- **cluster 0 (37%)** — mesma frequência de compra (8,5 transações contra 8,3), ticket baixo.
  Não é um cliente inativo: é um cliente de tíquete pequeno.
- **identidade ausente (13%)** — o que salta é `avg_ticket` de R$ 2,65 e gasto total de R$ 18,63,
  a metade do cluster 0. Cadastro incompleto anda junto com engajamento baixo.

Note que `view_rate` e `conv_rate` (colunas fora do ajuste) separam bem menos que o gasto: 0,65 a
0,76 de view, 0,46 a 0,54 de conversão. Os segmentos diferem muito mais em **quanto gastam** do
que em **se respondem**.

## 8. Resposta por segmento

Como cada segmento responde a cada tipo de oferta. É a leitura que a narrativa de resultados vai
retomar depois que o modelo estimar o efeito causal.

In [29]:
resposta = eda.segment_response(processed, spark.createDataFrame(
    segmentos[["account_id", "segmento"]]))
resposta.round(4)

,segmento,offer_type,envios,vistos,conversoes,receita,custo,taxa_view,taxa_conversao,taxa_conversao_vistos,margem_por_envio
0,cluster 0,bogo,11176,8599,2743,27990.88,17360.0,0.7694,0.2454,0.3190,0.9512
1,cluster 0,discount,11383,6422,1719,23125.11,4363.0,0.5642,0.1510,0.2677,1.6483
2,cluster 0,informational,5614,3292,3347,14356.60,0.0,0.5864,0.5962,1.0167,2.5573
3,cluster 1,bogo,15361,12665,10389,238525.98,77425.0,0.8245,0.6763,0.8203,10.4877
4,cluster 1,discount,15281,11019,10641,276130.83,30042.0,0.7211,0.6964,0.9657,16.1042
5,cluster 1,informational,7686,5235,3597,71891.81,0.0,0.6811,0.4680,0.6871,9.3536
6,identidade ausente,bogo,3962,3250,364,3920.58,2010.0,0.8203,0.0919,0.1120,0.4822
7,identidade ausente,discount,3879,2821,140,4133.13,368.0,0.7272,0.0361,0.0496,0.9706
8,identidade ausente,informational,1935,1351,1064,2821.58,0.0,0.6982,0.5499,0.7876,1.4582


In [30]:
melhor = resposta.loc[resposta["margem_por_envio"].idxmax()]
pior = resposta.loc[resposta["margem_por_envio"].idxmin()]

# Uma barra por tipo de oferta, agrupadas por segmento.
tipos = sorted(resposta["offer_type"].unique())
margem_wide = (
    resposta.pivot(index="segmento", columns="offer_type", values="margem_por_envio")
    .reset_index()
)
viz.grouped_bars(
    margem_wide, category="segmento", series=tipos, value_label="%{y:.2f}", ylabel="margem por envio",
    title=f"Margem por envio vai de {pior['margem_por_envio']:.2f} ({pior['segmento']}, {pior['offer_type']}) "
          f"a {melhor['margem_por_envio']:.2f} ({melhor['segmento']}, {melhor['offer_type']})",
    subtitle="Receita atribuída menos custo do desconto, por envio. Observacional: não é o efeito causal de enviar.",
)

A margem por envio varia de **−0,96** (BOGO para o segmento de identidade ausente) a **+36,53**
(`discount` no cluster 1) — quase quarenta reais de amplitude sobre a mesma decisão binária de
enviar ou não. O único par com margem negativa é BOGO × identidade ausente: o desconto de BOGO é
alto (5 ou 10 no catálogo) e esse segmento compra pouco, então o custo do cupom supera a receita
que ele traz. Uma política de envio uniforme paga essa conta.

A ordenação também muda por segmento, e é isso que torna o problema interessante: no cluster 1 e
no cluster 0 o melhor tipo é `discount`; no segmento de identidade ausente o melhor tipo passa a
ser `discount` também, mas `informational` — que não custa nada — bate BOGO com folga. Não existe
"a melhor oferta", existe a melhor oferta para cada cliente.

Atenção ao que este número **não** é: margem por envio observada mistura o efeito da oferta com o
fato de que clientes de alto valor compram muito de qualquer jeito. É a distinção que a próxima
célula torna explícita e que só o uplift resolve.

In [31]:
janela = eda.window_spend(attributed, events)
lift_bruto = eda.naive_spend_lift(processed, janela, spark.createDataFrame(
    segmentos[["account_id", "segmento"]]))
lift_bruto.round(2)

,segmento,gasto_visto,gasto_nao_visto,envios_vistos,envios_nao_vistos,diferenca_bruta
0,cluster 1,48.61,36.62,28919.0,9409.0,11.99
1,cluster 0,11.89,7.19,18313.0,9860.0,4.69
2,identidade ausente,5.17,4.17,7422.0,2354.0,1.00


`diferenca_bruta` compara o gasto na janela de validade entre quem viu e quem não viu a mesma
oferta — **sem** filtrar por view, ao contrário do label. Não é uplift: ver a oferta é escolha do
cliente, e quem abre tende a ser quem já estava mais ativo. A seleção inteira está dentro desse
número.

Serve para duas coisas honestas. Primeiro, mostrar que a resposta varia por segmento — é a
heterogeneidade que o X-learner vai modelar. Segundo, fixar a ordem de grandeza: se a estimativa
causal da spec 02 vier **maior** que esta diferença confundida, é sinal de erro no estimador, não
de descoberta.

## 9. Diagnóstico do desenho experimental

Duas perguntas que decidem se é possível ler efeito causal do dado observado: os grupos são
comparáveis, e todo cliente teve chance de receber todo tipo de oferta?

In [32]:
frame_balanco = eda.covariate_balance(processed, cfg)
limiar = cfg.smd_threshold
desbalanceadas = frame_balanco[frame_balanco["acima_do_limiar"]]["covariavel"].tolist()
conclusao = (
    f"Ver a oferta não é aleatório: {len(desbalanceadas)} covariável(is) acima de |SMD| {limiar}"
    if desbalanceadas else
    f"Tratado e controle são comparáveis: nenhuma covariável acima de |SMD| {limiar}"
)
# SMD infinito (variância nula, médias distintas) não é plotável — vai nomeado no subtítulo.
separa_perfeitamente = frame_balanco.loc[~np.isfinite(frame_balanco["smd"]), "covariavel"].tolist()
subtitulo = ("SMD entre quem viu e quem não viu a oferta. "
             "Diagnóstico que qualifica a leitura causal — não muda o estimador.")
if separa_perfeitamente:
    subtitulo += f" |SMD|=∞ (separa os grupos por completo): {', '.join(separa_perfeitamente)}."

viz.markers_with_thresholds(
    frame_balanco[np.isfinite(frame_balanco["smd"])],
    value="smd", label="covariavel", flag="acima_do_limiar",
    thresholds=(-limiar, limiar), title=conclusao, subtitle=subtitulo,
    xlabel=f"SMD (linhas pontilhadas: ±{limiar})",
)

In [33]:
frame_atribuicao = eda.assignment_balance(processed, cfg)
frame_atribuicao.round(4)

,covariavel,pior_abs_smd,acima_do_limiar
0,tenure_days,0.0394,False
1,gender=O,0.0347,False
2,gender=M,0.0325,False
3,identity_missing,0.0266,False
4,gender=unknown,0.0266,False
5,gender=F,0.0234,False
6,age,0.0218,False
7,credit_card_limit,0.0189,False


Duas leituras diferentes, ambas passando:

- **viu × não viu** (`covariate_balance`, o que o REQ-109 pede): nenhuma covariável acima de
  |SMD| 0,1. Tranquilizador, mas *ver* é comportamento pós-tratamento — este balanço não valida a
  aleatorização.
- **entre ofertas recebidas** (`assignment_balance`): o pior par de ofertas, na pior covariável,
  fica bem abaixo do limiar. É isto que verifica a Premissa 4 (o envio foi aleatorizado).

Balanço é diagnóstico, não gate: acima do limiar a leitura causal ficaria qualificada, o estimador
não mudaria (Premissa 5).

In [34]:
eda.treatment_group_comparison(processed).round(4)

,treatment,recebidos,taxa_conversao,ticket_medio,receita_media
0,não viu,21623,0.3302,19.6469,6.4875
1,viu,54654,0.4915,19.4542,9.5623


Comparação bruta viu × não viu, mesma leitura do balanço acima: taxa de conversão e ticket médio (só entre convertidos) por grupo. Não é efeito causal — é o número que justifica por que o pipeline segue para uplift em vez de comparar médias direto.

In [35]:
frame_positividade = eda.positivity_by_offer_type(processed, profile)
frame_positividade.assign(cobertura=lambda d: (100 * d["cobertura"]).round(1))

,offer_type,clientes_total,receberam,nunca_receberam,cobertura
0,bogo,17000,14992,2008,88.2
1,discount,17000,14945,2055,87.9
2,informational,17000,10547,6453,62.0


Nenhum tipo de oferta chegou a todos os clientes: `bogo` e `discount` cobriram ~88% da base,
`informational` apenas 62%. Sem sobreposição completa não há contrafactual observável para todo par (cliente, tipo):
IPW avalia bem políticas como "enviar a todos" ou "aleatória", mas o valor absoluto de negar um
tipo específico a um cliente que nunca o recebeu não é estimável sem premissa adicional
(Premissa 8).

In [36]:
frame_heterog = eda.conversion_by_type_and_segment(processed, profile, cfg)
ordem_q = ["Q1 (mais novo)", "Q2", "Q3", "Q4 (mais antigo)"]

maior = frame_heterog.loc[frame_heterog["taxa_conversao"].idxmax()]
menor = frame_heterog.loc[frame_heterog["taxa_conversao"].idxmin()]
# informational pode zerar num quartil; a razão viraria inf e o título mentiria.
razao = (f"{maior['taxa_conversao'] / menor['taxa_conversao']:.1f}×"
         if menor["taxa_conversao"] > 0 else "muitas vezes")

viz.line_series(
    frame_heterog, x="tenure_q", y="taxa_conversao", group="offer_type",
    category_order=ordem_q, mode="lines+markers", tickformat=".0%",
    xlabel="quartil de tenure", ylabel="taxa de conversão",
    title=f"Conversão varia {razao} entre tipo × quartil de tenure — há sinal heterogêneo para o uplift aprender",
    subtitle="Taxa de conversão influence-aware sobre os recebimentos (denominador: ofertas recebidas).",
)

In [37]:
frame_heterog.round(4)

,offer_type,tenure_q,n,vistos,conversoes,taxa_conversao,taxa_view,taxa_conversao_vistos
0,bogo,Q1 (mais novo),7601,6121,2115,0.2783,0.8053,0.3455
1,bogo,Q2,7685,6123,2842,0.3698,0.7967,0.4642
2,bogo,Q3,7517,6063,4513,0.6004,0.8066,0.7444
3,bogo,Q4 (mais antigo),7696,6207,4026,0.5231,0.8065,0.6486
4,discount,Q1 (mais novo),7786,5121,1974,0.2535,0.6577,0.3855
5,discount,Q2,7601,5047,2640,0.3473,0.6640,0.5231
6,discount,Q3,7517,5042,4184,0.5566,0.6707,0.8298
7,discount,Q4 (mais antigo),7639,5052,3702,0.4846,0.6613,0.7328
8,informational,Q1 (mais novo),3768,2387,1565,0.4153,0.6335,0.6556
9,informational,Q2,3818,2456,1820,0.4767,0.6433,0.7410


A tabela traz os três denominadores: `taxa_conversao` (sobre recebidos), `taxa_view` e
`taxa_conversao_vistos` (sobre vistos), e vale a identidade `taxa_conversao = taxa_view ×
taxa_conversao_vistos`, porque converter exige ver (G3). Tenure separa a conversão de forma
consistente em todos os tipos — é candidata a feature de alta importância no X-learner.
`informational` converte sistematicamente menos: é a oferta mais próxima de inócua.

## 10. Por que uplift e não classificação

O enquadramento do problema não é uma preferência metodológica; sai dos números acima.

In [38]:
frame_espontanea = eda.unattributable_transaction_share(events, attributed)
frac_fora = frame_espontanea.set_index("grupo").loc["fora de qualquer janela de oferta", "fracao"]
viz.vertical_bars(
    frame_espontanea, category="grupo", value="transacoes", ylabel="transações",
    label_template="%{y:,.0f}", hovertemplate="%{x}<br>%{y:,} transações<extra></extra>",
    title=f"{frac_fora:.1%} das compras não têm nenhuma oferta ativa por perto",
    subtitle="Transações fora de toda janela de recebimento, vista ou não — mais amplo que o filtro do label.",
)

In [39]:
distrib = processed.groupBy("treatment", "converted").count().toPandas()
distrib["grupo"] = distrib.apply(
    lambda r: {(1, 1): "viu e converteu", (1, 0): "viu e não converteu",
               (0, 1): "não viu e converteu", (0, 0): "não viu e não converteu"}[(r.treatment, r.converted)], axis=1)
distrib["fracao"] = distrib["count"] / distrib["count"].sum()
distrib[["grupo", "count", "fracao"]].round(3)

,grupo,count,fracao
0,não viu e converteu,7140,0.094
1,viu e converteu,26864,0.352
2,viu e não converteu,27790,0.364
3,não viu e não converteu,14483,0.190


`treatment = 0` com `converted = 1` é vazio **por construção** (G3: converter exige ver), não por
achado empírico.

Os quatro quadrantes causais, que o dataset não rotula:

| quadrante | comportamento | evidência aqui |
|---|---|---|
| **persuadable** | a oferta causou a compra | é o que o uplift procura |
| **sure thing** | compraria de qualquer jeito | 25,8% dos `completed` sem view (seção 3) |
| **lost cause** | não compra em hipótese alguma | oferta não muda nada |
| **sleeping dog** | a oferta o afasta | não observável sem teste dedicado |

Um classificador de completion premia igualmente *persuadable* e *sure thing* — ou seja, aprende
a mandar cupom para quem já ia comprar. Somando os 25,8% de completions sem view com o fato de que
uma fatia grande das transações acontece fora de qualquer janela de oferta, a canibalização não é
detalhe de borda: é o modo de falhar dominante. A pergunta certa é "para quem a oferta muda o
comportamento", e só incrementalidade responde.

## 11. Síntese

Cada linha é uma decisão do projeto e o número desta análise que a sustenta. Nada digitado à mão:
os valores vêm das células acima.

In [40]:
sintese = pd.DataFrame([
    ("Atribuição temporal oferta→transação",
     f"{n_txn_com_ref} de {n_txn:,} transações carregam offer_id", "Premissa 1 / REQ-103"),
    ("Label influence-aware (não usa `offer completed`)",
     f"{taxa_geral:.1%} dos completed sem view precedente", "Premissa 2 / REQ-104"),
    ("Label filtra pela janela de validade",
     f"{n_apos_validade:,} de {n_total_pares:,} completed após valid_until", "G4 / REQ-104"),
    ("Conversão de informational via janela pós-view",
     f"{int(frame_sem_view.set_index('offer_type').loc['informational', 'completados'])} eventos completed",
     "G5 / REQ-104"),
    ("Segmento identity_missing, nunca imputado",
     f"{int(frame_nulos.set_index('conjunto').loc['os três, juntos', 'clientes']):,} clientes, "
     "sobreposição perfeita nos três campos", "Premissa 3 / REQ-102"),
    ("Conversão exige o gasto mínimo da oferta",
     f"{frame_minimo['abaixo_do_minimo'].sum():,} conversões pagas abaixo do min_value "
     f"(auditoria de G10: deve ser 0)",
     "G10 / REQ-103 / REQ-106"),
    ("log1p + z-score antes do K-Means",
     f"p99/mediana de hist_spend_total = "
     f"{perfil_num.set_index('coluna').loc['hist_spend_total', 'p99'] / max(perfil_num.set_index('coluna').loc['hist_spend_total', 'p50'], 1):.0f}×",
     "REQ-111"),
    ("Segmentos são cortes num gradiente, não espécies",
     f"silhouette máxima {scan['silhouette'].max():.3f} em k={k}", "REQ-111"),
    ("Política de envio deve variar por segmento",
     f"margem/envio de {resposta['margem_por_envio'].min():.2f} a {resposta['margem_por_envio'].max():.2f}",
     "REQ-111 / spec 02"),
    ("Leitura causal direta se sustenta (sem ajuste)",
     f"pior |SMD| entre ofertas recebidas = {frame_atribuicao['pior_abs_smd'].max():.3f} "
     f"(limiar {cfg.smd_threshold})", "Premissas 4 e 5 / REQ-109"),
    ("IPW limitado pela positividade",
     f"cobertura mínima por tipo = {100 * frame_positividade['cobertura'].min():.0f}% da base",
     "Premissa 8 / spec 02"),
    ("Uplift, não classificação de completion",
     f"{frame_espontanea.set_index('grupo').loc['fora de qualquer janela de oferta', 'fracao']:.1%} "
     "das compras fora de toda janela de oferta", "Escopo / 00-clarify"),
    ("Split de validação respeita a onda",
     f"{len(frame_ondas)} disparos discretos em t={', '.join(str(int(t)) for t in frame_ondas['received_time'])}",
     "spec 02"),
    ("Censura das últimas ondas é artefato, não comportamento",
     f"{int(censura.set_index('janela_truncada').loc[1, 'count']):,} recebimentos "
     f"({100 * censura.set_index('janela_truncada').loc[1, 'count'] / n_linhas:.1f}%) com validade além de t={fim_do_teste}",
     "spec 02"),
    ("Atribuição é regra, não fato: ofertas concorrem",
     f"{100 * concorrencia.loc[concorrencia.n_concurrent_offers > 0, 'fracao'].sum():.1f}% dos "
     "recebimentos com outra oferta ativa na janela", "Premissa 1 / REQ-103"),
], columns=["decisão", "evidência", "onde vive"])
sintese

,decisão,evidência,onde vive
0,Atribuição temporal oferta→transação,"0 de 138,953 transações carregam offer_id",Premissa 1 / REQ-103
1,Label influence-aware (não usa `offer completed`),25.8% dos completed sem view precedente,Premissa 2 / REQ-104
2,Label filtra pela janela de validade,"4,685 de 33,182 completed após valid_until",G4 / REQ-104
3,Conversão de informational via janela pós-view,0 eventos completed,G5 / REQ-104
4,"Segmento identity_missing, nunca imputado","2,175 clientes, sobreposição perfeita nos três...",Premissa 3 / REQ-102
5,Conversão exige o gasto mínimo da oferta,0 conversões pagas abaixo do min_value (audito...,G10 / REQ-103 / REQ-106
6,log1p + z-score antes do K-Means,p99/mediana de hist_spend_total = 17×,REQ-111
7,"Segmentos são cortes num gradiente, não espécies",silhouette máxima 0.278 em k=2,REQ-111
8,Política de envio deve variar por segmento,margem/envio de 0.48 a 16.10,REQ-111 / spec 02
9,Leitura causal direta se sustenta (sem ajuste),pior |SMD| entre ofertas recebidas = 0.039 (li...,Premissas 4 e 5 / REQ-109


### Cinco divergências entre spec e dado, registradas

1. A **Premissa 2** cita 28,4% de `completed` sem view; a medição direta dá 25,8%. Nenhuma das
   definições que testamos reproduz 28,4%. A decisão (label influence-aware) não muda; o número da
   premissa deveria ser corrigido.
2. A **Premissa 1** ("uma oferta ativa por vez, por cliente") é falsa em 56,7% dos recebimentos.
   O pipeline já trata a disputa por regra explícita e logada, mas a premissa como escrita descreve
   um dataset que não é este. Reescrever a premissa, não o código.
3. **Resolvida em código (G10).** O **REQ-106** cobrava `reward_cost` em toda conversão de
   bogo/discount, inclusive nas que ficavam abaixo do `min_value` — 25,9% das conversões pagas.
   A atribuição passou a exigir `txn_amount >= min_value`: as conversões caem de 37.899 para 26.246
   e o custo de R$ 177.618 para R$ 107.779. O limiar é **por transação**, não sobre o gasto
   acumulado na janela.
4. **Censura à direita**: 13,4% dos recebimentos têm validade além do fim dos dados (t = 29,75).
   A conversão das ondas 4–5 é subestimada por construção; `campaign_wave` como feature aprenderia
   o artefato de coleta.
5. O empate de `received_time` (mesmo cliente, duas ofertas no mesmo instante) **não ocorre** no dado
   real — o desempate estável por `offer_id` é defensivo, coberto só por fixture sintética.

In [41]:
spark.stop()